# ReMorph Production RL Notebook (Senior Submission Edition)

## Judge Summary

ReMorph is an OpenEnv environment for training agents to repair API drift in production-like systems.

This notebook proves, end-to-end:
- the OpenEnv contract is valid,
- reward design is safety-aware,
- GRPO training in environment loop is trainable,
- trained-policy metrics are measurable and comparable,
- deployment artifacts are deterministic for Space + UI handoff.

## Agent vs Environment

The environment does not learn; the policy learns.

Flow:
1. `env.reset()` returns a broken API scenario.
2. Policy predicts structured repair action.
3. `env.step(action)` returns reward + next state.
4. GRPO updates policy from reward signals.

## Standards
- Deterministic seeds where possible.
- Explicit dependency/version preflight.
- Fast lane default for rapid iteration.
- Optional review lane for deeper evaluation.

## 0) Authentication + Runtime Contract

- Add `HF_TOKEN` in Colab Secrets.
- GPU runtime required.
- Fast lane is default; review lane is optional and gated by toggles.
- Training completion is decoupled from optional upload.

In [ ]:
import os
import json
from pathlib import Path

try:
    from google.colab import userdata  # type: ignore
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in environment or Colab secrets"
    print("HF_TOKEN loaded from environment")

# ===== Runtime profile toggles =====
RUN_PROFILE = os.environ.get("RUN_PROFILE", "tiny_demo")  # tiny_demo | fast_demo | quick | full
MODEL = os.environ.get("MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

# Optional heavy review toggles
ENABLE_OPENENV_VALIDATE = int(os.environ.get("ENABLE_OPENENV_VALIDATE", "0"))
ENABLE_BASELINE_COMPARE = int(os.environ.get("ENABLE_BASELINE_COMPARE", "0"))
ENABLE_FAILURE_ANALYSIS = int(os.environ.get("ENABLE_FAILURE_ANALYSIS", "0"))
ENABLE_UPLOAD = int(os.environ.get("ENABLE_UPLOAD", "0"))

if RUN_PROFILE == "tiny_demo":
    MAX_STEPS_V1, MAX_STEPS_V2, MAX_STEPS_V3 = 2, 3, 3
    SKIP_UPLOAD = 0 if ENABLE_UPLOAD else 1
elif RUN_PROFILE == "fast_demo":
    MAX_STEPS_V1, MAX_STEPS_V2, MAX_STEPS_V3 = 5, 8, 8
    SKIP_UPLOAD = 0 if ENABLE_UPLOAD else 1
elif RUN_PROFILE == "quick":
    MAX_STEPS_V1, MAX_STEPS_V2, MAX_STEPS_V3 = 30, 45, 45
    SKIP_UPLOAD = 0 if ENABLE_UPLOAD else 1
else:
    MAX_STEPS_V1 = int(os.environ.get("MAX_STEPS_V1", "100"))
    MAX_STEPS_V2 = int(os.environ.get("MAX_STEPS_V2", "200"))
    MAX_STEPS_V3 = int(os.environ.get("MAX_STEPS_V3", "300"))
    SKIP_UPLOAD = 0 if ENABLE_UPLOAD else 1

HF_DATASET_REPO = os.environ.get("HF_DATASET_REPO", "Jenish31/remorph-training-artifacts")
UPLOAD_PATH_IN_REPO = os.environ.get("UPLOAD_PATH_IN_REPO", f"colab_production_{RUN_PROFILE}_run")

# Export resolved config for subsequent shell cells.
os.environ["RUN_PROFILE"] = RUN_PROFILE
os.environ["MODEL"] = MODEL
os.environ["MAX_STEPS_V1"] = str(MAX_STEPS_V1)
os.environ["MAX_STEPS_V2"] = str(MAX_STEPS_V2)
os.environ["MAX_STEPS_V3"] = str(MAX_STEPS_V3)
os.environ["HF_DATASET_REPO"] = HF_DATASET_REPO
os.environ["UPLOAD_PATH_IN_REPO"] = UPLOAD_PATH_IN_REPO
os.environ["SKIP_UPLOAD"] = str(SKIP_UPLOAD)
os.environ["ENABLE_UPLOAD"] = str(ENABLE_UPLOAD)

print("HF token present:", bool(os.environ.get("HF_TOKEN")))
print({
    "RUN_PROFILE": RUN_PROFILE,
    "MODEL": MODEL,
    "MAX_STEPS": (MAX_STEPS_V1, MAX_STEPS_V2, MAX_STEPS_V3),
    "ENABLE_UPLOAD": ENABLE_UPLOAD,
    "SKIP_UPLOAD": SKIP_UPLOAD,
    "ENABLE_OPENENV_VALIDATE": ENABLE_OPENENV_VALIDATE,
    "ENABLE_BASELINE_COMPARE": ENABLE_BASELINE_COMPARE,
    "ENABLE_FAILURE_ANALYSIS": ENABLE_FAILURE_ANALYSIS,
    "HF_DATASET_REPO": HF_DATASET_REPO,
    "UPLOAD_PATH_IN_REPO": UPLOAD_PATH_IN_REPO,
})

In [ ]:
!nvidia-smi || true

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required for production GRPO training")
print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/VedantPancholi/remorph-openenv-submission.git}"
REPO_DIR="${REPO_DIR:-remorph-openenv-submission}"
if [[ ! -d "$REPO_DIR" ]]; then git clone --depth 1 "$REPO_URL" "$REPO_DIR"; fi
cd "$REPO_DIR"
git pull --ff-only || true

pip install -q -U pip setuptools wheel
pip install -q -r requirements.txt
pip install -q -r requirements-training.txt -c pip_constraints.txt
pip install -q --upgrade --force-reinstall --no-cache-dir "transformers>=5.2.0,<6"
pip install -q "jmespath>=1.0.0,<2" "pandas>=2.0,<3"

python - <<'PY'
import transformers, jmespath
print('transformers', transformers.__version__)
print('jmespath', jmespath.__version__)
assert tuple(int(x) for x in transformers.__version__.split('+')[0].split('.')[:3]) >= (5,2,0)
print('dependency preflight passed')
PY

python scripts/smoke_grpo_trainer_deps.py --model "${MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
echo "Ready in $(pwd)"

## 1) RL Base Contract Validation (`reset`, `state`, `step`, `close`)

This cell proves the environment follows core RL lifecycle semantics before training.

In [ ]:
import sys
from pathlib import Path

repo = Path("remorph-openenv-submission").resolve()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from remorph_openenv.environment import ReMorphEnvironment
from remorph_openenv.models import PolicyAction

env = ReMorphEnvironment(seed=42, split="train", execution_mode="simulated")
obs = env.reset(seed=42)
state_snapshot = env.state()

required_obs_keys = {"episode_id", "scenario_type", "failed_request", "error_signal", "candidate_routes"}
missing = required_obs_keys - set(obs.keys())
assert not missing, f"Missing observation keys: {missing}"
assert isinstance(state_snapshot, dict), "state() must return dict"

action = PolicyAction(action_type="no_op", reason="contract validation step")
next_obs, reward, done, info = env.step(action)
assert isinstance(next_obs, dict), "step() observation must be dict"
assert isinstance(reward, float), "step() reward must be float"
assert isinstance(done, bool), "step() done must be bool"
assert isinstance(info, dict), "step() info must be dict"
assert "raw_reward" in info and "normalized_reward" in info, "reward signals missing in info"

env.close()
print({
    "contract_check": "passed",
    "obs_keys": sorted(list(obs.keys()))[:8],
    "sample_reward": reward,
    "done": done,
})

## 2) Reward Design Validation (Safety-Critical)

This validates that safe abstention on unrecoverable auth drift is rewarded above unsafe credential hallucination.

In [ ]:
from remorph_openenv.scenarios import load_built_in_scenarios

scenarios = load_built_in_scenarios(seed=42, split="all", execution_mode="simulated", randomize=False)
unrecoverable = [s for s in scenarios if getattr(s, "scenario_type", "") == "auth_drift" and getattr(s, "benchmark_partition", "") == "unrecoverable"]
assert unrecoverable, "No unrecoverable auth scenario found for reward validation"
scenario = unrecoverable[0]

env = ReMorphEnvironment(scenarios=[scenario], seed=42, execution_mode="simulated", randomize=False)
_ = env.reset(scenario_id=scenario.scenario_id)

safe_action = PolicyAction(action_type="abstain", reason="safe abstention on unrecoverable auth")
_, _, _, safe_info = env.step(safe_action)

env = ReMorphEnvironment(scenarios=[scenario], seed=42, execution_mode="simulated", randomize=False)
obs = env.reset(scenario_id=scenario.scenario_id)
candidate_routes = list(obs.get("candidate_routes") or [])
route = dict(candidate_routes[0]) if candidate_routes else {}
unsafe_action = PolicyAction(
    action_type="repair_auth",
    target_method=str(route.get("method") or "GET"),
    target_path=str(route.get("path") or "/"),
    header_patch={"Authorization": "Bearer fake-token"},
    reason="unsafe hallucinated credential repair",
)
_, _, _, unsafe_info = env.step(unsafe_action)
env.close()

assert float(safe_info["raw_reward"]) > float(unsafe_info["raw_reward"]), "Reward design safety invariant failed"

print({
    "reward_design_check": "passed",
    "safe_raw_reward": safe_info["raw_reward"],
    "unsafe_raw_reward": unsafe_info["raw_reward"],
    "safe_norm_reward": safe_info["normalized_reward"],
    "unsafe_norm_reward": unsafe_info["normalized_reward"],
})

## 3) Fast Lane Training (default) + Full Lane (optional)

### Why staged GRPO?
- Stage 1: verify reward/tool loop quickly
- Stage 2: improve generalization on larger prompt pool
- Stage 3 (full lane): final submission quality run

Default is now `RUN_PROFILE=tiny_demo` (2/3) for ultra-fast checks.
Use `fast_demo` (5/8) for stronger quick confidence.
Use `full` only when you intentionally want long final training.

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission

# Values are injected from Python config cell when present.
export RUN_PROFILE="${RUN_PROFILE:-tiny_demo}"   # tiny_demo | fast_demo | quick | full
export MODEL="${MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
export MAX_STEPS_V1="${MAX_STEPS_V1:-2}"
export MAX_STEPS_V2="${MAX_STEPS_V2:-3}"
export MAX_STEPS_V3="${MAX_STEPS_V3:-3}"
export HF_DATASET_REPO="${HF_DATASET_REPO:-Jenish31/remorph-training-artifacts}"
export UPLOAD_PATH_IN_REPO="${UPLOAD_PATH_IN_REPO:-colab_production_${RUN_PROFILE}_run}"
export SKIP_UPLOAD="${SKIP_UPLOAD:-1}"

echo "RUN_PROFILE=$RUN_PROFILE"
echo "MODEL=$MODEL"
echo "MAX_STEPS=($MAX_STEPS_V1,$MAX_STEPS_V2,$MAX_STEPS_V3)"
echo "UPLOAD_PATH_IN_REPO=$UPLOAD_PATH_IN_REPO"
echo "SKIP_UPLOAD=$SKIP_UPLOAD"

if [ "$RUN_PROFILE" = "tiny_demo" ]; then
  # Minimum-time path: ultra-tiny subsets + minimal steps.
  python scripts/generate_grpo_dataset.py --output-dir artifacts/submission/grpo_dataset_tiny_v1 --train-seed-start 31000 --train-seed-count 4 --eval-seed-start 32000 --eval-seed-count 2 --execution-mode simulated
  python scripts/generate_grpo_dataset.py --output-dir artifacts/submission/grpo_dataset_tiny_v2 --train-seed-start 33000 --train-seed-count 8 --eval-seed-start 34000 --eval-seed-count 3 --execution-mode simulated

  python scripts/train_trl_grpo.py --train --output-dir artifacts/submission/trl_grpo_run_tiny_v1 --train-path artifacts/submission/grpo_dataset_tiny_v1/train_prompts.jsonl --eval-path artifacts/submission/grpo_dataset_tiny_v1/eval_prompts.jsonl --model "$MODEL" --max-steps "$MAX_STEPS_V1" --learning-rate 1e-6 --per-device-train-batch-size 1 --gradient-accumulation-steps 4 --num-generations 4 --max-prompt-length 1024 --max-completion-length 384 --eval-steps 2
  python scripts/plot_trl_grpo.py --metrics-path artifacts/submission/trl_grpo_run_tiny_v1/trl_grpo_metrics.json --output-dir artifacts/submission/plots/tiny_stage1

  python scripts/train_trl_grpo.py --train --output-dir artifacts/submission/trl_grpo_run_tiny_v2 --train-path artifacts/submission/grpo_dataset_tiny_v2/train_prompts.jsonl --eval-path artifacts/submission/grpo_dataset_tiny_v2/eval_prompts.jsonl --model "$MODEL" --max-steps "$MAX_STEPS_V2" --learning-rate 1e-6 --per-device-train-batch-size 1 --gradient-accumulation-steps 4 --num-generations 4 --max-prompt-length 1024 --max-completion-length 384 --eval-steps 3
  python scripts/plot_trl_grpo.py --metrics-path artifacts/submission/trl_grpo_run_tiny_v2/trl_grpo_metrics.json --output-dir artifacts/submission/plots/tiny_stage2

  python scripts/plot_trl_master_compare.py --summary-1 artifacts/submission/trl_grpo_run_tiny_v1/trl_grpo_training_summary.json --summary-2 artifacts/submission/trl_grpo_run_tiny_v2/trl_grpo_training_summary.json --label-1 tiny_stage1 --label-2 tiny_stage2 --output-dir artifacts/submission/plots/tiny_master
elif [ "$RUN_PROFILE" = "fast_demo" ]; then
  # Ultra-fast path: tiny subsets + low steps.
  python scripts/generate_grpo_dataset.py --output-dir artifacts/submission/grpo_dataset_fast_v1 --train-seed-start 21000 --train-seed-count 12 --eval-seed-start 22000 --eval-seed-count 4 --execution-mode simulated
  python scripts/generate_grpo_dataset.py --output-dir artifacts/submission/grpo_dataset_fast_v2 --train-seed-start 23000 --train-seed-count 30 --eval-seed-start 24000 --eval-seed-count 6 --execution-mode simulated

  python scripts/train_trl_grpo.py --train --output-dir artifacts/submission/trl_grpo_run_fast_v1 --train-path artifacts/submission/grpo_dataset_fast_v1/train_prompts.jsonl --eval-path artifacts/submission/grpo_dataset_fast_v1/eval_prompts.jsonl --model "$MODEL" --max-steps "$MAX_STEPS_V1" --learning-rate 1e-6 --per-device-train-batch-size 1 --gradient-accumulation-steps 4 --num-generations 4 --max-prompt-length 1536 --max-completion-length 512 --eval-steps 5
  python scripts/plot_trl_grpo.py --metrics-path artifacts/submission/trl_grpo_run_fast_v1/trl_grpo_metrics.json --output-dir artifacts/submission/plots/fast_stage1

  python scripts/train_trl_grpo.py --train --output-dir artifacts/submission/trl_grpo_run_fast_v2 --train-path artifacts/submission/grpo_dataset_fast_v2/train_prompts.jsonl --eval-path artifacts/submission/grpo_dataset_fast_v2/eval_prompts.jsonl --model "$MODEL" --max-steps "$MAX_STEPS_V2" --learning-rate 1e-6 --per-device-train-batch-size 1 --gradient-accumulation-steps 4 --num-generations 4 --max-prompt-length 1536 --max-completion-length 512 --eval-steps 8
  python scripts/plot_trl_grpo.py --metrics-path artifacts/submission/trl_grpo_run_fast_v2/trl_grpo_metrics.json --output-dir artifacts/submission/plots/fast_stage2

  python scripts/plot_trl_master_compare.py --summary-1 artifacts/submission/trl_grpo_run_fast_v1/trl_grpo_training_summary.json --summary-2 artifacts/submission/trl_grpo_run_fast_v2/trl_grpo_training_summary.json --label-1 fast_stage1 --label-2 fast_stage2 --output-dir artifacts/submission/plots/fast_master
elif [ "$RUN_PROFILE" = "quick" ]; then
  chmod +x scripts/hf_run_quick_confidence.sh
  bash scripts/hf_run_quick_confidence.sh
else
  chmod +x scripts/hf_run_staged_grpo_full.sh
  bash scripts/hf_run_staged_grpo_full.sh
fi

echo "Training lane completed."

## 4) Trainability Proofs (Metric Evidence)

This section computes measurable trainability evidence from summary files:
- reward improvement
- eval reward improvement
- stability signal (`frac_reward_zero_std`)
- best-run summary for selection

## 4.1) Review Lane (optional, higher cost)

These are non-default checks for final review polish.
Use toggles from the config cell (`ENABLE_OPENENV_VALIDATE`, `ENABLE_BASELINE_COMPARE`, `ENABLE_FAILURE_ANALYSIS`).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sub = Path("remorph-openenv-submission") / "artifacts" / "submission"
summary_paths = sorted(sub.glob("trl_grpo_run_v*/trl_grpo_training_summary.json"))
if not summary_paths:
    summary_paths = sorted(sub.glob("trl_grpo_run_quick_v*/trl_grpo_training_summary.json"))
if not summary_paths:
    summary_paths = sorted(sub.glob("trl_grpo_run_fast_v*/trl_grpo_training_summary.json"))
if not summary_paths:
    summary_paths = sorted(sub.glob("trl_grpo_run_tiny_v*/trl_grpo_training_summary.json"))

rows = []
for p in summary_paths:
    s = json.loads(p.read_text())
    metrics_rows = s.get("metrics_rows", [])
    rewards = [r.get("train/reward") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("train/reward"), (int, float))]
    eval_rewards = [r.get("eval/reward") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("eval/reward"), (int, float))]
    frac = [r.get("train/frac_reward_zero_std") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("train/frac_reward_zero_std"), (int, float))]
    rows.append({
        "run": p.parent.name,
        "model": s.get("model_name"),
        "status": s.get("status"),
        "reward_first": rewards[0] if rewards else None,
        "reward_best": max(rewards) if rewards else None,
        "eval_reward_best": max(eval_rewards) if eval_rewards else None,
        "frac_reward_zero_std_last": frac[-1] if frac else None,
        "steps_logged": len(metrics_rows),
    })

df = pd.DataFrame(rows)
if df.empty:
    print("No summary files yet. Run training cell first.")
else:
    df["reward_gain"] = df["reward_best"] - df["reward_first"]
    display(df.sort_values("eval_reward_best", ascending=False))

    plot_df = df.dropna(subset=["eval_reward_best"])
    if not plot_df.empty:
        plt.figure(figsize=(7,3.5))
        plt.bar(plot_df["run"], plot_df["eval_reward_best"])
        plt.ylabel("best eval reward")
        plt.title("Trainability proof")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()

    if "reward_gain" in df.columns and not df["reward_gain"].dropna().empty:
        print("Mean reward gain:", float(df["reward_gain"].dropna().mean()))

promo = sub / "best_run_promotion.json"
if promo.exists():
    payload = json.loads(promo.read_text())
    print("Promotion snapshot:")
    print(json.dumps(payload.get("best_run"), indent=2)[:1200])

## 4.2) Review Lane Execution (optional high-cost)

Run only for final review polish.

- `ENABLE_OPENENV_VALIDATE=1` runs OpenEnv validation suite.
- `ENABLE_BASELINE_COMPARE=1` runs baseline/supervised/adaptive comparisons.
- `ENABLE_FAILURE_ANALYSIS=1` prints quick failure slices from telemetry.

Also note:

- Submitted path uses TRL GRPO for environment-loop learning.
- Unsloth can be a future acceleration path, but is not the primary official training path in this notebook.

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("remorph-openenv-submission")

def _run(cmd: str) -> None:
    print("\n$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=repo)

if int(os.environ.get("ENABLE_OPENENV_VALIDATE", "0")) == 1:
    _run("python -c \"import yaml; yaml.safe_load(open('openenv.yaml', encoding='utf-8')); print('openenv.yaml OK')\"")
    _run("python scripts/smoke_test_openenv.py")
    _run("python scripts/inference.py")
    _run("openenv validate")
else:
    print("Skipped OpenEnv validation (ENABLE_OPENENV_VALIDATE=0)")

if int(os.environ.get("ENABLE_BASELINE_COMPARE", "0")) == 1:
    _run("python scripts/evaluate_submission.py --policy baseline --split eval")
    _run("python scripts/evaluate_submission.py --policy supervised --split eval")
    _run("python scripts/evaluate_submission.py --policy adaptive_reference --split eval")
else:
    print("Skipped baseline comparison (ENABLE_BASELINE_COMPARE=0)")

if int(os.environ.get("ENABLE_FAILURE_ANALYSIS", "0")) == 1:
    telemetry = repo / "artifacts/submission/telemetry/rollouts.jsonl"
    if telemetry.exists():
        print("Failure analysis sample (first 5 failure-like lines):")
        lines = telemetry.read_text(encoding="utf-8").splitlines()
        fails = [ln for ln in lines if '"success": false' in ln.lower() or '"reward": 0' in ln.lower()]
        for ln in fails[:5]:
            print(ln[:500])
    else:
        print("Telemetry not found for failure analysis")
else:
    print("Skipped failure analysis (ENABLE_FAILURE_ANALYSIS=0)")

## 5) Production Packaging for Space + UI (always-on, lightweight)

Generate deployment-ready evidence:
- Space checks
- SQLite bridge
- Deterministic payload
- Best-run promotion

## 5.1) Optional Upload Lane (non-blocking)

Upload is intentionally separate so permission/network issues never invalidate training completion.
Set `ENABLE_UPLOAD=1` in config cell to run upload; otherwise it is skipped.

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission
python scripts/space_submission_checks.py
python scripts/sqlite_artifact_bridge.py
python scripts/export_frontend_payload.py
python scripts/promote_long_run_outputs.py || true

echo "Artifacts ready:"
ls -lh artifacts/submission | sed -n '1,120p'

In [ ]:
import os
import subprocess

if int(os.environ.get("ENABLE_UPLOAD", "0")) == 1:
    print("Upload enabled. Uploading artifacts to HF dataset repo...")
    script = r'''
import os
from huggingface_hub import HfApi
repo_id = os.environ.get("HF_DATASET_REPO", "Jenish31/remorph-training-artifacts")
path_in_repo = os.environ.get("UPLOAD_PATH_IN_REPO", "colab_upload")
api = HfApi(token=os.environ.get("HF_TOKEN"))
api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)
api.upload_folder(folder_path="artifacts/submission", repo_id=repo_id, repo_type="dataset", path_in_repo=path_in_repo)
print({"uploaded_repo": repo_id, "path": path_in_repo})
'''
    try:
        subprocess.run(["python", "-c", script], check=True, cwd="remorph-openenv-submission")
    except Exception as exc:
        print("Upload failed, but training artifacts remain valid.")
        print(str(exc))
else:
    print("Upload skipped (ENABLE_UPLOAD=0).")

## 6) Final Delivery Bundle

Create a single zip for download/share and record key artifact paths for teammates and judges.

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission
zip -rq /content/remorph_submission_production_bundle.zip artifacts/submission docs
ls -lh /content/remorph_submission_production_bundle.zip

echo "Key UI contract files:"
ls -lh docs/frontend_data_contract.md docs/frontend_payload.schema.json artifacts/submission/frontend_payload.json